In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

In [2]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [134]:
class double_conv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch, momentum=0.005),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch, momentum=0.005),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv(x)
        return x
    
class single_conv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch, momentum=0.005),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = self.conv(x)
        return x


class inconv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.conv = double_conv(in_ch, out_ch)

    def forward(self, x):
        x = self.conv(x)
        return x


class down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.mpconv = nn.Sequential(
            nn.MaxPool2d(2),
            double_conv(in_ch, out_ch)
        )

    def forward(self, x):
        x = self.mpconv(x)
        return x


class up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = double_conv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x2, x1], dim=1)
        x = self.conv(x)
        return x
    
class up2(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = double_conv(out_ch * 2, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x2, x1], dim=1)
        x = self.conv(x)
        return x


class outconv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super(type(self), self).__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x):
        x = self.conv(x)
        return x

In [135]:
class UNet2(nn.Module):
    def __init__(self, n_channels, n_classes, hidden=32):
        super(type(self), self).__init__()
        self.inc = inconv(n_channels, hidden)
        self.down1 = down(hidden, hidden * 2)
        self.up8 = up(hidden * 2, hidden)
        self.outc = outconv(hidden, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x = self.up8(x2, x1)
        x = self.outc(x)
        return torch.sigmoid(x)
    
class UNet3(nn.Module):
    def __init__(self, n_channels, n_classes, hidden=32):
        super(type(self), self).__init__()
        self.inc = inconv(n_channels, hidden)
        self.down1 = down(hidden, hidden * 2)
        self.down2 = down(hidden * 2, hidden * 4)
        self.up7 = up(hidden * 4, hidden * 2)
        self.up8 = up(hidden * 2, hidden)
        self.outc = outconv(hidden, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x = self.up7(x3, x2)
        x = self.up8(x, x1)
        x = self.outc(x)
        return torch.sigmoid(x)
    
class UNet4(nn.Module):
    def __init__(self, n_channels, n_classes, hidden=32):
        super(type(self), self).__init__()
        self.inc = inconv(n_channels, hidden)
        self.down1 = down(hidden, hidden * 2)
        self.down2 = down(hidden * 2, hidden * 4)
        self.down3 = down(hidden * 4, hidden * 8)
        self.up6 = up(hidden * 8, hidden * 4)
        self.up7 = up(hidden * 4, hidden * 2)
        self.up8 = up(hidden * 2, hidden)
        self.outc = outconv(hidden, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up6(x4, x3)
        x = self.up7(x, x2)
        x = self.up8(x, x1)
        x = self.outc(x)
        return torch.sigmoid(x)
    
class UNet5(nn.Module):
    def __init__(self, n_channels, n_classes, hidden=32):
        super(type(self), self).__init__()
        self.inc = inconv(n_channels, hidden)
        self.down1 = down(hidden, hidden * 2)
        self.down2 = down(hidden * 2, hidden * 4)
        self.down3 = down(hidden * 4, hidden * 8)
        self.down4 = down(hidden * 8, hidden * 16)
        self.up5 = up(hidden * 16, hidden * 8)
        self.up6 = up(hidden * 8, hidden * 4)
        self.up7 = up(hidden * 4, hidden * 2)
        self.up8 = up(hidden * 2, hidden)
        self.outc = outconv(hidden, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up5(x5, x4)
        x = self.up6(x, x3)
        x = self.up7(x, x2)
        x = self.up8(x, x1)
        x = self.outc(x)
        return torch.sigmoid(x)
    
    
class RNN_UNet5(nn.Module):
    """
    Combination of U-net and a RNN to do object segmentation in a time sequence of images.
    First a encoder is used to extract features from all the images. In the bottle neck a RNN 
    (GRU or LSTM) is used to relate the features as a sequence. The output of the RNN is used 
    as input to the decoder side to expand and reconstruct the image mask using skip connections 
    from the decoder side. 
    
    Image data input has shape of [batch_size, n_times, H_size, W_size]
    Output is the probability mask with shape [batch_size, n_times, H_size, W_size]. 
    
    """
    def __init__(self, n_channels, n_classes, hidden=32, rnn_size=1024, rnn_hidden=1024):
        super(type(self), self).__init__()
        self.inc = inconv(n_channels, hidden)
        self.down1 = down(hidden, hidden * 2)
        self.down2 = down(hidden * 2, hidden * 4)
        self.down3 = down(hidden * 4, hidden * 8)
        self.down4 = down(hidden * 8, hidden * 16)
        self.up5 = up(hidden * 16, hidden * 8)
        self.up6 = up(hidden * 8, hidden * 4)
        self.up7 = up(hidden * 4, hidden * 2)
        self.up8 = up(hidden * 2, hidden)
        self.outc = outconv(hidden, n_classes)
        
        self.rnn = nn.GRU(rnn_size, rnn_hidden, num_layers=1, batch_first=True)

    def forward(self, x):
        # not sure if I can skip this for loops and make it more efficient
        # we need to save the ouput of each conv block for the skip connections
        nt = x.shape[1]
        x_1 = []
        x_2 = []
        x_3 = []
        x_4 = []
        x_5 = []
        for k in range(nt):
            x1 = network.inc(Xt[:, k, :, :].view(33, 1, 64, 64))
            x2 = network.down1(x1)
            x3 = network.down2(x2)
            x4 = network.down3(x3)
            x5 = network.down4(x4)
            x_1.append(x1)
            x_2.append(x2)
            x_3.append(x3)
            x_4.append(x4)
            down4_out = x5.shape
            # flattent the image to a vector
            x_5.append(x5.flatten(start_dim=1, end_dim=-1).unsqueeze(1))
            
        # concat all times to have shape [batch, time, features], this is the input
        # to the RNN
        rnn_input = torch.cat(x_5, dim=1)
        rnn_ouput, rnn_h = self.rnn(rnn_input)

        x_out = []
        for k in range(nt):
            # need to reshape the rnn feature output to image shape [batch, time, h, w]
            x = network.up5(rnn_ouput[:, k, :].view(down4_out), x_4[k])
            x = network.up6(x, x_3[k])
            x = network.up7(x, x_2[k])
            x = network.up8(x, x_1[k])
            x = network.outc(x)
            x_out.append(x)

        # concat the time dimension and take sigmoid to get [0, 1] prob values
        return torch.sigmoid(torch.cat(x_out, dim=1))
    
class WrappedModel(nn.Module):
    def __init__(self, network):
        super(type(self), self).__init__()
        self.module = network

    def forward(self, *x):
        return self.module(*x)

In [152]:
class R_UNet5_block(nn.Module):
    """
    Unet with two inputs, image and segmentaion mask. Intended to be trained recursevely 
    in time to get a time series of segmentation masks.
    """
    def __init__(self, n_channels, n_classes, hidden=4):
        super(type(self), self).__init__()
        
        self.inc_imag = inconv(n_channels, hidden)
        self.inc_mask = inconv(n_channels, hidden)
        
        # we multiply the number of channels x2 for the image side to
        # account for the concat of the image and mask
        self.down11 = down(hidden * 2, hidden * 2)
        self.down12 = down(hidden, hidden * 2)
        
        self.down21 = down(hidden * 2 * 2, hidden * 4)
        self.down22 = down(hidden * 2, hidden * 4)
        
        self.down31 = down(hidden * 4 * 2, hidden * 8)
        self.down32 = down(hidden * 4, hidden * 8)
        
        self.down41 = down(hidden * 8 * 2, hidden * 16)
        self.down42 = down(hidden * 8, hidden * 16)
        
        self.up5 = up2(hidden * 16 * 2, hidden * 8)
        self.up6 = up2(hidden * 8 * 2, hidden * 4)
        self.up7 = up2(hidden * 4 * 2, hidden * 2)
        self.up8 = up2(hidden * 2 * 2, hidden)
        
        self.outc_mask = outconv(hidden, n_classes)
        self.outc_gate = nn.Sequential(
            single_conv(hidden, hidden), 
            outconv(hidden, n_classes)
        )

    def forward(self, x, m):
        x1 = self.inc_imag(x)
        m1 = self.inc_mask(m)
        xm1 = torch.cat([x1, m1], dim=1)
        
        x2 = self.down11(xm1)
        m2 = self.down12(m1)
        xm2 = torch.cat([x2, m2], dim=1)
        
        x3 = self.down21(xm2)
        m3 = self.down22(m2)
        xm3 = torch.cat([x3, m3], dim=1)
        
        x4 = self.down31(xm3)
        m4 = self.down32(m3)
        xm4 = torch.cat([x4, m4], dim=1)
        
        x5 = self.down41(xm4)
        m5 = self.down42(m4)
        xm5 = torch.cat([x5, m5], dim=1)
        
        x = self.up5(xm5, x4)

        x = torch.cat([x, m4], dim=1) 
        x = self.up6(x, x3)
        
        x = torch.cat([x, m3], dim=1)
        x = self.up7(x, x2)

        x = torch.cat([x, m2], dim=1)
        x = self.up8(x, x1)
        
        out_x = self.outc_mask(x)
        out_gate = self.outc_gate(x)
        
        return torch.sigmoid(out_x), torch.sigmoid(out_gate)

In [125]:
hidden = 4

In [126]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [153]:
network = R_UNet5_block(1, 1, hidden)
network.to(device)
network.type(torch.FloatTensor)
print(count_parameters(network))
# network

232274


In [34]:
from torchview import draw_graph

In [38]:
sector = 4
camera = 1
ccd = 3
cutout_size = 64

In [39]:
PACKAGEDIR = "/Users/jorgemarpa/Work/BAERI/ADAP/tess-asteroid-ml/"
out_path = f"{os.path.dirname(PACKAGEDIR)}/data/asteroidcuts/sector{sector:04}"
out_file = f"{out_path}/tess-asteroid-cuts_{cutout_size}x{cutout_size}_s{sector:04}-{camera}-{ccd}_01.npz"
out_file

'/Users/jorgemarpa/Work/BAERI/ADAP/tess-asteroid-ml/data/asteroidcuts/sector0004/tess-asteroid-cuts_64x64_s0004-1-3_01.npz'

In [141]:
data = np.load(out_file)
data

NpzFile '/Users/jorgemarpa/Work/BAERI/ADAP/tess-asteroid-ml/data/asteroidcuts/sector0004/tess-asteroid-cuts_64x64_s0004-1-3_01.npz' with keys: flux, column, row, mask, time...

In [142]:
X = data["flux"]
y = data["mask"]
# (n_cutout, n_times, h_size, w_size)
X.shape, y.shape

((33, 1027, 64, 64), (33, 1027, 64, 64))

In [143]:
Xt = torch.from_numpy(X)
yt = torch.from_numpy(y).float()

In [144]:
Mt = torch.rand(yt.shape)
Mt.shape

torch.Size([33, 1027, 64, 64])

In [163]:
x, m = Xt[:, 1, :, :], Mt[:, 1, :, :]

In [165]:
x.unsqueeze(1).shape

torch.Size([33, 1, 64, 64])

In [146]:
x1 = network.inc_imag(x)
m1 = network.inc_mask(m)
xm1 = torch.cat([x1, m1], dim=1)

x2 = network.down11(xm1)
m2 = network.down12(m1)
xm2 = torch.cat([x2, m2], dim=1)

x3 = network.down21(xm2)
m3 = network.down22(m2)
xm3 = torch.cat([x3, m3], dim=1)

x4 = network.down31(xm3)
m4 = network.down32(m3)
xm4 = torch.cat([x4, m4], dim=1)

x5 = network.down41(xm4)
m5 = network.down42(m4)
xm5 = torch.cat([x5, m5], dim=1)

In [147]:
x5.shape, m5.shape, xm5.shape

(torch.Size([33, 64, 4, 4]),
 torch.Size([33, 64, 4, 4]),
 torch.Size([33, 128, 4, 4]))

In [148]:
network.up5

up2(
  (up): ConvTranspose2d(128, 32, kernel_size=(2, 2), stride=(2, 2))
  (conv): double_conv(
    (conv): Sequential(
      (0): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.005, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.005, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
)

In [149]:
x = network.up5(xm5, x4)

x = torch.cat([x, m4], dim=1) 
x = network.up6(x, x3)

x = torch.cat([x, m3], dim=1)
x = network.up7(x, x2)

x = torch.cat([x, m2], dim=1)
x = network.up8(x, x1)

out_x = network.outc_mask(x)
out_gate = network.outc_gate(x)

In [154]:
pred_mask, gates = network(Xt[:, :1, :, :], Mt[:, :1, :, :])
pred_mask.shape, gates.shape

(torch.Size([33, 1, 64, 64]), torch.Size([33, 1, 64, 64]))

In [158]:
pred_mask

torch.Size([33, 1, 64, 64])

In [168]:
class LambdaLayer(nn.Module):
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd
    def forward(self, x):
        return self.lambd(x)
    
class GaussianNoise(nn.Module):
    def __init__(self, stddev):
        super().__init__()
        self.stddev = stddev

    def forward(self, din):
        if self.training:
            return din + torch.autograd.Variable(torch.randn(din.size()).cuda() * self.stddev)
        return din

In [175]:
class Rec_UNet5(nn.Module):
    """
    Unet with two inputs, image and segmentaion mask. Intended to be trained recursevely 
    in time to get a time series of segmentation masks.
    """
    def __init__(self, n_channels, n_classes, hidden=4, nt=100, add_noise=False):
        super(type(self), self).__init__()
        
        self.unet_block = R_UNet5_block(n_channels, n_classes, hidden)
        self.lambda_malyer = LambdaLayer(lambda x: 1 - x)
        self.add_noise = add_noise
        self.gaussian_noise = GaussianNoise(0.1)
        self.nt = nt

    def forward(self, x):
        
        mask_sequence = []
        mask_sequence.append(torch.rand((x.shape[0], 1, x.shape[2], x.shape[3])))
        for k in range(self.nt):
            out_seg, out_gate = self.unet_block(
                x[:, k, :, :].unsqueeze(1), 
                mask_sequence[-1]
            )
            
            if add_noise:
                noisy_seg = self.gaussian_noise(out_seg)
            else:
                noisy_seg = out_seg
            
            flipped_gate = self.lambda_malyer(out_gate)
            new_mask = torch.add(
                torch.multiply(out_seg, out_gate),
                torch.multiply(noisy_seg, flipped_gates)
            )
            mask_sequence.append(new_mask)
            
        mask_sequence = torch.cat(mask_sequence[1:], dim=1)
        
        return mask_sequence

In [173]:
Xt.shape

torch.Size([33, 1027, 64, 64])

In [178]:
recurrent_unet = Rec_UNet5(1, 1, hidden=4, nt=1027)
recurrent_unet

Rec_UNet5(
  (unet_block): R_UNet5_block(
    (inc_imag): inconv(
      (conv): double_conv(
        (conv): Sequential(
          (0): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): BatchNorm2d(4, eps=1e-05, momentum=0.005, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (4): BatchNorm2d(4, eps=1e-05, momentum=0.005, affine=True, track_running_stats=True)
          (5): ReLU(inplace=True)
        )
      )
    )
    (inc_mask): inconv(
      (conv): double_conv(
        (conv): Sequential(
          (0): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): BatchNorm2d(4, eps=1e-05, momentum=0.005, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(4, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (4): BatchNorm2d(4, eps=1e-05, momentum=0.005, affine=True, 